# Figure 6: Storm Xaver, model versus tide gauges

(a) Observed versus modelled surge peaks at the tide gauges; (b) surge time series at one station, with error statistics for each experiment.

From Özkan et al. (2025), *Sensitivity of global storm surge modelling to sea surface drag*, Ocean Dynamics, https://doi.org/10.1007/s10236-025-01713-3

The processed input files are not included in this repository; this notebook shows how the figure was made. Experiment names: in the code, `cc` = FIX, `cv` = NST, `ntr`/`uv10n` = CAD and `rho`/`uv10rho` = IFS (see the main README).

In [ ]:
# Folder with the processed input files (not included in this repository)
DATA_DIR = "path/to/paper1_data/"

In [ ]:
import os
import xarray as xr
import numpy as np
import proplot as pplt
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import cartopy.feature as cfeature
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import gaussian_kde
import matplotlib.lines as mlines
from scipy.stats import probplot
import matplotlib.dates as mdates

In [ ]:
data_base_dir  = DATA_DIR + "data/4-Processed_for_Visualization/"

In [ ]:
# Define storm details
storms = {
    "Sandy": {
        "landfall": "2012-10-30 00:00:00",
        "analysis_time": ["2012-10-27 00:00:00", "2012-10-31 00:00:00"],
        "filter_lon_lat_limits": [-80, -60, 30, 50],
        "g3_lon_lat_limits": [-75.8, -70, 35, 42],
        "plot_lon_0": -74, "plot_lat_0": 40,
        "plot_lon_lat_limits": [-76.01, -69.90, 37.95, 42], # lonmin, lonmax, latmin, latmax
        "lonlocator": [-76, -74, -72, -70],
        "latlocator": [38, 39, 40, 41, 42],
        "levels_surge": np.arange(0, 2.6, 0.1),
        "ticks_surge": [0, 0.5, 1, 1.5, 2, 2.5],
        "station_to_drop" : ['Inside_Thorofare_At_Atlantic_City_Nj'],
    },
    "Xaver": {
        "landfall": "2013-12-06 05:00:00",
        "analysis_time": ["2013-12-04 00:00:00", "2013-12-08 00:00:00"],
        "g3_lon_lat_limits": [2, 10, 51.30, 56],
        "plot_lon_lat_limits": [1.99, 9.05, 51.20, 56.01], # lonmin, lonmax, latmin, latmax
        "plot_lon_0": 5, "plot_lat_0": 56,
        "lonlocator": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        "latlocator": [49, 50, 51, 52, 53, 54, 55, 56],
        "levels_surge": np.arange(0, 3.6, 0.1),
        "ticks_surge": [0, 0.5, 1, 1.5, 2, 2.5, 3.0, 3.5],
        "station_to_drop" : [],
        #"station_to_drop" : ['Roompot_binnen'],
        "station_to_plot" : ['Bake_C_Scharhorn', 'Brouwershavensche_Gat_08', 'Cadzand', 'Den_Helder', 
                             'Euro_platform', 'Haringvliet_10', 'Hoek_van_Holland',
                             'Huibertgat', 'Ijmuiden_buitenhaven', 'Kornwerderzand_buiten',
                             'Lichteiland_Goeree', 'Petten_zuid', 'Scheveningen',
                             'Terschelling_Noordzee', 'Vlieland_haven', 'West_Terschelling',
                             'Wierumergronden', "Helgoland"],
    },
    "Halong": {
        "landfall": "2014-08-10 03:00:00",
        "analysis_time": ["2014-08-08 00:00:00", "2014-08-12 00:00:00"],
        "filter_lon_lat_limits": [125, 145, 30, 50],
        "g3_lon_lat_limits": [133, 136, 33, 35],
        "plot_lon_lat_limits": [132.99, 136.01, 32.99, 35.01], # lonmin, lonmax, latmin, latmax
        "plot_lon_0": 135, "plot_lat_0": 34,
        "lonlocator": [133, 134, 135, 136],
        "latlocator": [33, 34, 35,],
        "levels_surge": np.arange(0, 0.85, 0.05),
        "ticks_surge": np.arange(0, 0.9, 0.1),
        "station_to_drop" : [],
    },
    "Grayson": {
        "landfall": "2018-01-05 11:00:00",
        "analysis_time": ["2018-01-03 00:00:00", "2018-01-07 00:00:00"],
        "filter_lon_lat_limits": [-74, -50, 36, 55],
        "g3_lon_lat_limits": [-72, -60, 40, 52],
        "plot_lon_lat_limits": [-72.01, -59.90, 39.90, 50.01], # lonmin, lonmax, latmin, latmax
        "plot_lon_0": -64, "plot_lat_0": 46,
        "lonlocator": [-72, -70, -68, -66, -64, -62, -60,],
        "latlocator": [40, 42, 44, 46, 48, 50,],
        "levels_surge": np.arange(-0.2, 1.25, 0.05),
        "ticks_surge": np.arange(-0.2, 1.25, 0.1),
        "station_to_drop" : [],
    },
    "Jebi": {
        "landfall": "2018-09-04 06:00:00",
        "analysis_time": ["2018-09-02 00:00:00", "2018-09-06 00:00:00"],
        "filter_lon_lat_limits": [125, 145, 30, 50],
        "g3_lon_lat_limits": [133, 137, 32, 36],
        "plot_lon_lat_limits": [132.99, 137.01, 32.99, 36.01], # lonmin, lonmax, latmin, latmax
        "plot_lon_0": 135, "plot_lat_0": 34,
        "lonlocator": [133, 134, 135, 136, 137],
        "latlocator": [33, 34, 35, 36],
        "levels_surge": np.arange(0, 1.25, 0.05),
        "ticks_surge": np.arange(0, 1.3, 0.1),
        "station_to_drop" : ['Kumano', "Tajiri", "Tannowa", "Uragami"],
    },
    "Fiona": {
        "nc_read": "202209220000_202209260000",
        "landfall": "2022-09-24 09:00:00",
        "analysis_time": ["2022-09-22 00:00:00", "2022-09-26 00:00:00"],
        "filter_lon_lat_limits": [-74, -50, 36, 55],
        "g3_lon_lat_limits": [-70, -60, 40, 52],
        "plot_lon_lat_limits": [-68.05, -59.90, 42.95, 49.05], # lonmin, lonmax, latmin, latmax
        "plot_lon_0": -62, "plot_lat_0": 46,
        "lonlocator": [-68, -66, -64, -62, -60],
        "latlocator": [42, 44, 46, 48, 50, 52],
        "levels_surge": np.arange(0, 2.01, 0.1),
        "ticks_surge": [0, 0.2, 0.4, 0.6, 0.8, 1, 1.2, 1.4, 1.6, 1.8, 2],
        "station_to_drop" : [],
        "station_to_plot" : ['Lower_escuminac', 'North_sydney'],
    }
}

In [ ]:
storm_to_process = "Xaver"
data_dir = os.path.join(data_base_dir, "1_storm_cases/", storm_to_process)
# Load the data
ds_gtsm = xr.open_dataset(os.path.join(data_dir, f"{storm_to_process}_surge_his.nc"))
ds_obs = xr.open_dataset(os.path.join(data_dir, f"{storm_to_process}_surge_his_obs.nc")) 
print(f"Data loaded for storm {storm_to_process} and GESLA3...")

In [ ]:
# Convert 'station_name' to a coordinate
ds_obs = ds_obs.set_coords('station_name')
ds_gtsm = ds_gtsm.set_coords('station_name')
                    
# Ensure 'stations' is the dimension name in the dataset, if not you can use .isel method
selected_stations = storms[storm_to_process]["station_to_plot"]
indexed_ds_obs = ds_obs.sel(stations=ds_obs['station_name'].isin(selected_stations))
indexed_ds_gtsm = ds_gtsm.sel(stations=ds_gtsm['station_name'].isin(selected_stations))

In [ ]:
# Station shown in the time series (panel b)
st = 11

# Extract the surge data
obs_surge = ds_obs.surge[st]
cc_surge = ds_gtsm.cc_surge[st]
cv_surge = ds_gtsm.cv_surge[st]
ntr_surge = ds_gtsm.ntr_surge[st]
rho_surge = ds_gtsm.rho_surge[st]

In [ ]:
# Define the data
obs_max = ds_obs.surge.max(dim="time")
cc_max = ds_gtsm.cc_surge.max(dim="time")
cv_max = ds_gtsm.cv_surge.max(dim="time")
ntr_max = ds_gtsm.ntr_surge.max(dim="time")
rho_max = ds_gtsm.rho_surge.max(dim="time")
time_series = indexed_ds_obs.time.values

In [ ]:
# Helper function to calculate RMSE and Mean Absolute Error (MAE)
def calculate_metrics(obs, pred):
    obs = np.ravel(obs)
    pred = np.ravel(pred)
    rmse = np.sqrt(mean_squared_error(obs, pred))
    mae = np.mean(np.abs(pred - obs))  # Mean Absolute Error
    return rmse, mae

In [ ]:
# Import necessary library
from sklearn.metrics import r2_score

# Helper function to calculate RMSE, MAE, and R^2
def calculate_metrics_with_r2(obs, pred):
    obs = np.ravel(obs)
    pred = np.ravel(pred)
    rmse = np.sqrt(mean_squared_error(obs, pred))
    mae = np.mean(np.abs(pred - obs))  # Mean Absolute Error
    r2 = r2_score(obs, pred)  # Coefficient of determination
    return rmse, mae, r2

In [ ]:
# Adjusted label box padding for tighter fit
fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1, 1.5]})  # Adjusted width ratio

# Plot 1: Scatter plot
ax1 = axes[0]
metrics_lines1 = []
for label, pred_surge in zip(['$GTSM_{IFS}$', '$GTSM_{CAD}$', '$GTSM_{NST}$', '$GTSM_{FIX}$'], 
                             [rho_max, ntr_max, cv_max, cc_max]):
    rmse, mae = calculate_metrics(obs_max, pred_surge)
    metrics_lines1.append(f'{label}: RMSE={rmse:.2f}, MAE={mae:.2f}')

metrics_text1 = '\n'.join(metrics_lines1)

# Dynamically calculate limits based on observed data
min_limit = obs_max.min() - 0.4
max_limit = obs_max.max() + 0.2
lims = [min_limit, max_limit]

# Scatter plot
ax1.scatter(obs_max, rho_max, label="$GTSM_{IFS}$", marker='d', color='rose red', s=100, edgecolor='black', alpha=0.8)
ax1.scatter(obs_max, ntr_max, label="$GTSM_{CAD}$", marker='^', color='emerald', s=100, edgecolor='black', alpha=0.8)
ax1.scatter(obs_max, cv_max, label="$GTSM_{NST}$", marker='s', color='pumpkin orange', s=100, edgecolor='black', alpha=0.8)
ax1.scatter(obs_max, cc_max, label="$GTSM_{FIX}$", marker='o', color='cerulean', s=100, edgecolor='black', alpha=0.8)


# 1:1 line
ax1.plot(lims, lims, 'k--', linewidth=1.2, label="Perfect fit")
ax1.set_xlim(lims)
ax1.set_ylim(lims)
ax1.set_xlabel("Observed surge peak (m)", fontsize=13)
ax1.set_ylabel("Model surge peak (m)", fontsize=13)
ax1.legend(loc="upper left", bbox_to_anchor=(0.00, 0.95), fontsize=12, frameon=True, ncol=2,
           columnspacing=0.5, handletextpad=0.5)
ax1.text(0.006, 0.783, metrics_text1, transform=ax1.transAxes, fontsize=13, verticalalignment='top',
         bbox=dict(boxstyle="square,pad=0.2", edgecolor="black", facecolor="white"))  # Reduced padding
ax1.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

# Set tick label size for Plot 1
ax1.tick_params(axis='both', which='major', labelsize=12)

# Add 'a' label to the top-left corner
ax1.text(
    0.007, 0.993, 'a', transform=ax1.transAxes, fontsize=14, fontweight='bold',
    verticalalignment='top', horizontalalignment='left',
    bbox=dict(boxstyle="square", edgecolor="black", facecolor="white", pad=0.2)  # Reduced padding
)

# Plot 2: Time series plot
ax2 = axes[1]
metrics_lines2 = []
for label, pred_surge in zip(['$GTSM_{IFS}$', '$GTSM_{CAD}$', '$GTSM_{NST}$', '$GTSM_{FIX}$'], 
                             [rho_surge, ntr_surge, cv_surge, cc_surge]):
    rmse, mae, r2 = calculate_metrics_with_r2(obs_surge, pred_surge)
    metrics_lines2.append(f'{label}: RMSE={rmse:.2f}, MAE={mae:.2f}, $R^2$={r2:.2f}')

metrics_text2 = '\n'.join(metrics_lines2)

ax2.plot(time_series, obs_surge, label='OBS', color='black', linestyle='--', linewidth=1.6)
ax2.plot(time_series, rho_surge, label='$GTSM_{IFS}$', color='rose red', linestyle='-', linewidth=1.3)
ax2.plot(time_series, ntr_surge, label='$GTSM_{CAD}$', color='emerald', linestyle='-', linewidth=1.3)
ax2.plot(time_series, cv_surge, label='$GTSM_{NST}$', color='pumpkin orange', linestyle='-', linewidth=1.3)
ax2.plot(time_series, cc_surge, label='$GTSM_{FIX}$', color='cerulean', linestyle='-', linewidth=1.3)
ax2.axvline(pd.to_datetime('2013-12-06 05:00 UTC'), color='hotpink', linestyle=':', linewidth=1.8)

# Format time axis
ax2.xaxis.set_major_locator(mdates.HourLocator(interval=24))
ax2.xaxis.set_minor_locator(mdates.HourLocator(interval=6))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %Y'))
ax2.set_xlim(time_series.min(), time_series.max())  # Set x-axis limits to the time range
ax2.set_xlabel("Time", fontsize=13)
ax2.set_ylabel("Surge height (m)", fontsize=13)
ax2.legend(loc="upper right", fontsize=13)
ax2.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)

# Set tick label size for Plot 2
ax2.tick_params(axis='both', which='major', labelsize=11)

# Add metrics text
ax2.text(0.0060, 0.864, metrics_text2, transform=ax2.transAxes, fontsize=11.8, verticalalignment='center',
         bbox=dict(boxstyle="square,pad=0.2", edgecolor="black", facecolor="white"))  # Reduced padding

# Add 'b' label to the top-left corner
ax2.text(
    0.007, 0.993, 'b', transform=ax2.transAxes, fontsize=14, fontweight='bold',
    verticalalignment='top', horizontalalignment='left',
    bbox=dict(boxstyle="square", edgecolor="black", facecolor="white", pad=0.2)  # Reduced padding
)

# Adjust layout
plt.tight_layout()

# Save the figure
output_path_png = "Xaver_surge_comparison_plots_2.png"
output_path_pdf = "Xaver_surge_comparison_plots_2.pdf"
plt.savefig(output_path_png, dpi=300, bbox_inches="tight")
plt.savefig(output_path_pdf, dpi=300, bbox_inches="tight")

# Show the plot
plt.show()